<a href="https://colab.research.google.com/github/pkocmann/messyverse/blob/main/notebooks/01_isbn-open-library.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab oeffnen"/></a>

# ISBN-Liste anreichern -- ueber die Open Library

Die Bibliothek hat eine Liste von ISBNs (`katalog/isbns.txt`), aber zu vielen Titeln fehlen Autor und Erscheinungsjahr. Die **Open Library** ist ein frei zugaenglicher Web-Katalog fuer Buchmetadaten. Dein Auftrag: zu jeder ISBN Titel, Autor und Jahr holen und eine Tabelle bauen.

Du schreibst keinen Code selbst. Du schaust die Lage an, sagst deinem KI-Assistenten praezise, was du willst, fuegst seinen Code ein, fuehrst ihn aus und **pruefst gegen das Universum**.

In [ ]:
# SETUP (Black Box -- einmal ausfuehren). Holt deine Arbeitskopie des Uebungsuniversums.
import os, shutil, subprocess
ZIEL = "/content/messyverse"
os.chdir("/content")                 # erst aus ZIEL heraus -- sonst loescht der Re-Lauf das eigene Arbeitsverzeichnis
if os.path.exists(ZIEL):
    shutil.rmtree(ZIEL)              # idempotent: alte Kopie weg, dann frisch klonen
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/pkocmann/messyverse.git", ZIEL], check=True)
os.chdir(ZIEL)
anzahl = sum(len(d) for r, _, d in os.walk(ZIEL) if ".git" not in r)
print(f"Arbeitskopie: {anzahl} Dateien unter {ZIEL}")

## Wir arbeiten gegen gespeicherte Antworten (fixtures-first)

Damit die Uebung fuer alle gleich und ohne Wartezeit laeuft, liegen die Antworten der Open Library schon gespeichert in `api-fixtures/openlibrary/` -- eine JSON-Datei pro ISBN. Der echte Live-Aufruf ist der Bonus am Ende. Der Vertrag steht in `api-fixtures/openlibrary/README.md`.

## Schritt 1 -- die Lage ansehen

Sieh dir an, womit du arbeitest: die ISBN-Liste und eine einzelne gespeicherte Antwort.

In [ ]:
print(open("katalog/isbns.txt").read())
import json
beispiel = json.load(open("api-fixtures/openlibrary/9783518111383.json"))
print(json.dumps(beispiel, ensure_ascii=False, indent=2)[:600])

## Schritt 2 -- die KI prompten

Ein brauchbarer Prompt nennt Datenquelle, Soll-Ergebnis und Ausgabeformat:

> *Lies `katalog/isbns.txt`. Lade fuer jede ISBN `api-fixtures/openlibrary/<isbn>.json`. Zieh aus dem Objekt unter `ISBN:<isbn>` den `title`, die Namen aus `authors` und `publish_date`. Baue ein dict `ergebnis`: ISBN -> {'titel':..., 'autoren':[...], 'jahr':...} und gib eine Tabelle aus.*

Lass die KI das Ergebnis genau `ergebnis` nennen -- die Pruefzelle sucht danach.

In [ ]:
# Code deines KI-Assistenten hier einfuegen und ausfuehren.
# Ziel: dict `ergebnis` -> ISBN: {'titel':..., 'autoren':[...], 'jahr':...}


## Schritt 3 -- gegen das Universum pruefen (Black Box)

Vergleicht dein `ergebnis` mit dem Soll und nennt, **wo** es klemmt -- ohne die Loesung zu verraten.

In [ ]:
import json
soll = json.load(open("loesungen/isbn_lookup.golden.json"))
try:
    ergebnis
except NameError:
    print("Es gibt noch kein `ergebnis`. Lass deinen Assistenten das dict bauen (Schritt 2).")
else:
    if not isinstance(ergebnis, dict):
        print(f"Dein `ergebnis` ist gerade ein {type(ergebnis).__name__}, erwartet wird ein dict ISBN -> {{...}}. Bitte deinen Assistenten, es als Python-dict zu bauen.")
    else:
        fehlen = [i for i in soll if i not in ergebnis]
        kein_obj = [i for i in soll if i in ergebnis and not isinstance(ergebnis[i], dict)]
        titel_ab = [i for i in soll if i in ergebnis and isinstance(ergebnis[i], dict) and ergebnis[i].get("titel") != soll[i]["titel"]]
        if not fehlen and not kein_obj and not titel_ab:
            print(f"OK -- alle {len(soll)} Titel stimmen mit dem Universum ueberein.")
        else:
            if fehlen: print("Diese ISBNs fehlen:", fehlen)
            if kein_obj: print("Diese Eintraege sind kein {...}-Objekt mit 'titel':", kein_obj)
            if titel_ab: print("Bei diesen ISBNs weicht der Titel ab:", titel_ab)

## Wenn es klemmt -- die Fehlerschleife

Lief der Code auf einen Fehler, oder meldet die Pruefzelle Abweichungen? Dann **nicht selbst reparieren**. Kopiere die Fehlermeldung oder die Abweichungs-Liste, gib sie deinem Assistenten zurueck und bitte um eine korrigierte Fassung. Die haeufigsten Ursachen sind uebersehene Sonderfaelle -- leere Eintraege, fehlende Felder, ungewohnte Schreibweisen.

## Bonus (live) -- die echte Open Library und die 302-Falle

Optional, mit Netz: Der Daten-Endpoint `https://openlibrary.org/api/books?bibkeys=ISBN:9783518111383&jscmd=data&format=json` antwortet direkt mit JSON. Die **302-Weiterleitung** zeigt sich am Buch-Endpoint `https://openlibrary.org/isbn/9783518111383` -- lass deine KI beide abrufen und die Statuscodes vergleichen (einmal ohne automatisches Folgen der Weiterleitung). Mit Pausen abrufen; 429/Timeout sind erwartbar.